In [1]:
import pandas as pd

df = pd.read_csv("../data/emails.csv")

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4198 entries, 0 to 4197
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   subject  4185 non-null   str  
 1   sender   4196 non-null   str  
 2   body     4190 non-null   str  
 3   label    4198 non-null   str  
dtypes: str(4)
memory usage: 12.8 MB


In [3]:
df["label"].value_counts()

label
ham     2801
spam    1397
Name: count, dtype: int64

In [4]:
df.isnull().sum()

subject    13
sender      2
body        8
label       0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(200)

In [6]:
duplicates = df[df.duplicated(
    subset=["subject", "body"],
    keep=False
)]

print("Duplicate emails:", len(duplicates))

print(
    duplicates[
        ["subject", "body", "label"]
    ].sort_values("subject").head(10)
)

Duplicate emails: 416
                                                subject  \
847   "Free" Elvis Costello CD a trojan horse for DR...   
848   "Free" Elvis Costello CD a trojan horse for DR...   
3276          $500,000 Life Policy $9.50 per month. vyw   
3382          $500,000 Life Policy $9.50 per month. vyw   
2781  (SPAM? 08.00) lists.sourceforge.net mailing li...   
2800  (SPAM? 08.00) lists.sourceforge.net mailing li...   
283                                          /home/dude   
1390                                         /home/dude   
180   10 die as Israeli helicopter fires on Palestin...   
2485  10 die as Israeli helicopter fires on Palestin...   

                                                   body label  
847   A friend in Dublin is mailing me the CD which ...   ham  
848   A friend in Dublin is mailing me the CD which ...   ham  
3276  <html>\n\n<body>\n\n<font size="2" PTSIZE="10"...  spam  
3382  <html>\n\n<body>\n\n<font size="2" PTSIZE="10"...  spam  
2781  **

In [7]:
label_conflicts = (
    df.groupby(["subject", "body"])["label"]
    .nunique()
)

print(
    "Emails with conflicting labels:",
    (label_conflicts > 1).sum()
)

Emails with conflicting labels: 0


In [8]:
df = df.drop_duplicates(
    subset=["subject", "body"]
).reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nRemaining duplicates:")
print(
    df.duplicated(
        subset=["subject", "body"]
    ).sum()
)

Shape after removing duplicates: (3979, 4)

Class distribution:
label
ham     2638
spam    1341
Name: count, dtype: int64

Remaining duplicates:
0


In [9]:
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

In [10]:
df["text"] = df["subject"] + " " + df["body"]

In [11]:
df["subject_length"] = df["subject"].str.len()
df["body_length"] = df["body"].str.len()
df["text_length"] = df["text"].str.len()

print("\nAverage lengths:")
print(
    df.groupby("label")[
        ["subject_length", "body_length", "text_length"]
    ].mean()
)

print("\nMedian lengths:")
print(
    df.groupby("label")[
        ["subject_length", "body_length", "text_length"]
    ].median()
)


Average lengths:
       subject_length  body_length  text_length
label                                          
ham         37.286960  2792.768385  2831.055345
spam        40.030574  3682.710664  3723.741238

Median lengths:
       subject_length  body_length  text_length
label                                          
ham              34.0        933.5        966.0
spam             37.0       1864.0       1897.0


In [12]:
from collections import Counter
import re


def get_words(text):
    return re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())


for label in ["ham", "spam"]:

    words = []

    for text in df[df["label"] == label]["text"]:
        words.extend(get_words(text))

    word_counts = Counter(words)

    print(f"\nTop 20 words in {label.upper()}:")

    print(
        word_counts.most_common(20)
    )


Top 20 words in HAM:
[('the', 31292), ('http', 23781), ('com', 23452), ('width', 15101), ('and', 14572), ('www', 12572), ('font', 9898), ('height', 8373), ('src', 8296), ('gif', 8219), ('img', 8164), ('that', 7699), ('for', 7639), ('href', 6959), ('cnet', 6105), ('border', 6074), ('you', 5921), ('table', 5221), ('this', 4873), ('size', 4794)]

Top 20 words in SPAM:
[('font', 28888), ('the', 13432), ('size', 9787), ('you', 9487), ('and', 9474), ('nbsp', 8954), ('color', 8152), ('face', 7750), ('http', 6543), ('your', 6173), ('com', 5563), ('width', 5498), ('for', 5488), ('arial', 5450), ('align', 4932), ('center', 4344), ('this', 4304), ('www', 3620), ('table', 3148), ('href', 3130)]


In [13]:
html_mask = df["body"].str.contains(
    r"<html|<body|<table|<font|<div|<br|<img",
    case=False,
    regex=True,
    na=False
)

print("HTML emails:", html_mask.sum())
print("Non-HTML emails:", (~html_mask).sum())

print("\nHTML by class:")
print(
    pd.crosstab(
        df["label"],
        html_mask,
        normalize="index"
    ) * 100
)

HTML emails: 814
Non-HTML emails: 3165

HTML by class:
body       False      True 
label                      
ham    94.996209   5.003791
spam   49.142431  50.857569


In [14]:
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

df["text"] = (
    df["subject"] + " " + df["body"]
).str.strip()

print("Missing text:", df["text"].isna().sum())

print("Empty emails:", (df["text"].str.len() == 0).sum())

print("\nText examples:")
print(df[["label", "subject", "text"]].head(3))

Missing text: 0
Empty emails: 1

Text examples:
  label                    subject  \
0   ham   Re: New Sequences Window   
1   ham  [zzzzteana] RE: Alexander   
2   ham  [zzzzteana] Moscow bomber   

                                                text  
0  Re: New Sequences Window     Date:        Wed,...  
1  [zzzzteana] RE: Alexander Martin A posted:\nTa...  
2  [zzzzteana] Moscow bomber Man Threatens Explos...  


In [15]:
df = df[df["text"].str.len() > 0].reset_index(drop=True)

print("Final dataset shape:", df.shape)
print(df["label"].value_counts())

Final dataset shape: (3978, 8)
label
ham     2638
spam    1340
Name: count, dtype: int64


In [16]:
df.to_csv("../data/cleaned_emails.csv", index=False)